In [2]:
import sys
if "ESP32_env" not in sys.executable:
    print("/n环境配置错误!!!/n")
    print(sys.executable)
else:
    print("环境配置正常")

环境配置正常


In [3]:
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt 
from matplotlib import rcParams # 字体配置,支持中文
rcParams['font.family'] = 'SimHei'
rcParams['axes.unicode_minus'] = False  # 解决负号不显示的问题

import tensorflow as tf

In [3]:
# ===== 1. 数据集构建与保存 =====

In [4]:
# 1-1 数据集构建
np.random.seed(42)

N = 1000    # 生成1000份数据

# 生成更真实的分布（博物馆大多数时间在正常附近）
temp = np.random.normal(23, 3, N)
humi = np.random.normal(50, 10, N)
gray = np.random.normal(0.2, 0.15, N)

# 限制物理合理范围
temp = np.clip(temp, 10, 40)
humi = np.clip(humi, 20, 90)
gray = np.clip(gray, 0, 1)

def evaluate(t, h, g):
    # 高风险
    if t < 18 or t > 28 or h < 35 or h > 65 or g > 0.5:
        return 2
    
    # 轻度风险
    if (18 <= t < 20 or 26 < t <= 28 or
        35 <= h < 40 or 60 < h <= 65 or
        0.3 < g <= 0.5):
        return 1
    
    # 安全
    return 0

risk = np.array([evaluate(t, h, g) for t, h, g in zip(temp, humi, gray)])

df = pd.DataFrame({
    "temp": temp,
    "humi": humi,
    "gray": gray,
    "risk": risk
})

df = df.round({
    "temp": 2,
    "humi": 2,
    "gray": 3
})

print(df.head())

    temp   humi   gray  risk
0  24.49  63.99  0.099     1
1  22.59  59.25  0.178     0
2  24.94  50.60  0.081     0
3  27.57  43.53  0.154     1
4  22.30  56.98  0.000     0


In [5]:
# 1-2 生成数据集
df.to_csv("museum_env_data.csv", index=False)

In [6]:
# ===== 2. 模型训练 =====

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 读取数据
df = pd.read_csv("museum_env_data.csv")

# 分离特征和标签
X = df[["temp", "humi", "gray"]].values
y = df["risk"].values

# 数据归一化
scaler = StandardScaler()
X = scaler.fit_transform(X)

# 划分训练和测试
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 建立模型
model = tf.keras.Sequential([
    tf.keras.layers.Dense(16, activation="relu", input_shape=(3,)), # type:ignore
    tf.keras.layers.Dense(3, activation="softmax")
])

# 编译模型
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# 训练
history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# 测试准确率
loss, acc = model.evaluate(X_test, y_test)
print("测试准确率:", acc)

d:\conda_envs\ESP32_env\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 7s 92ms/step - accuracy: 0.2781 - loss: 1.1223 - val_accuracy: 0.3063 - val_loss: 1.1264
Epoch 2/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 90ms/step - accuracy: 0.3469 - loss: 1.0794 - val_accuracy: 0.3250 - val_loss: 1.0939
Epoch 3/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 71ms/step - accuracy: 0.3984 - loss: 1.0450 - val_accuracy: 0.4062 - val_loss: 1.0701
Epoch 4/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 49ms/step - accuracy: 0.4688 - loss: 1.0211 - val_accuracy: 0.5312 - val_loss: 1.0493
Epoch 5/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.5500 - loss: 1.0014 - val_accuracy: 0.5562 - val_loss: 1.0326
Epoch 6/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.5922 - loss: 0.9850 - val_accuracy: 0.5688 - val_loss: 1.0190
Epoch 7/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.5938 - loss: 0.9716 - val_accuracy: 0.5625 - val_loss: 1.0062
Epoch 8/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.5859 - loss: 0.9598 - val_accuracy: 0.5688 - v

In [8]:
# ===== 3. 模型导出 =====

In [9]:
# 3-1 保存为 TFLite 模型
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("museum_model.tflite", "wb") as f:
    f.write(tflite_model)

print("TFLite 模型已生成")

INFO:tensorflow:Assets written to: C:\Users\lxl\AppData\Local\Temp\tmpc4amixu5\assets


INFO:tensorflow:Assets written to: C:\Users\lxl\AppData\Local\Temp\tmpc4amixu5\assets


Saved artifact at 'C:\Users\lxl\AppData\Local\Temp\tmpc4amixu5'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  2898958458256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2898958459408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2898958458064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2898958457296: TensorSpec(shape=(), dtype=tf.resource, name=None)
TFLite 模型已生成


In [10]:
# 3-2 转成C数组
with open("museum_model.tflite", "rb") as f:
    data = f.read()

with open("model_data.cc", "w") as f:
    f.write("const unsigned char model[] = {\n")
    for i, b in enumerate(data):
        f.write(f"0x{b:02x},")
        if i % 12 == 0:
            f.write("\n")
    f.write("\n};\n")
    f.write(f"const unsigned int model_len = {len(data)};")